In [18]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from som_rl.som.som_backend import SharpZSOMBackend, SharpZBackendConfig
from som_rl.eval.ideal_som import get_perfect_bins

DATA = "data"  # adjust if needed

LEVEL_MIN = 30
LEVEL_MAX = 200  # inclusive; must be < levels_available() once loaded

In [2]:
# Load SOM / correlation data and create backend (copied from 14-InteractiveInspector)

npz = np.load(f"{DATA}/correlations_dl/npairs_compact.npz")
dd, dr, rd, rr = npz["dd"], npz["dr"], npz["rd"], npz["rr"]

node_indices_photo = np.load(f"{DATA}/node_indices_fc.npy")
initial_node_bins  = np.load(f"{DATA}/nodes_labels_initial_fc_200_ext.npy")
reference_labels   = np.load(f"{DATA}/reference_labels_fc_fc.npy")
weighted_clusters  = np.load(f"{DATA}/weighted_clusters_5c_1t200.npy")
umatrix = np.load(f"{DATA}/umat.npy")

perfect_bins = get_perfect_bins()

b_cfg = SharpZBackendConfig(
    grid_shape=(200, 200),
    num_bins=11,
    objective_mode="energy",
    proposal_bin_mode="uniform_other",
)


def make_backend():
    return SharpZSOMBackend(
        cfg=b_cfg,
        dd=dd, dr=dr, rd=rd, rr=rr,
        node_indices_photo=node_indices_photo,
        initial_node_bins=initial_node_bins,
        reference_labels=reference_labels,
        umatrix=umatrix,
        weighted_clusters=weighted_clusters,
    )


backend = make_backend()
K = backend.K
H, W = backend.H, backend.W
L = backend.levels_available()
print(f"Backend ready: grid {H}x{W}, K={K}, levels={L}")

if LEVEL_MAX >= L:
    print(f"WARNING: LEVEL_MAX={LEVEL_MAX} >= levels_available={L}, clamping.")
    LEVEL_MAX = L - 1

Backend ready: grid 200x200, K=11, levels=201


In [3]:
def compute_deltaE_per_bin(backend, node, level, perfect_bins):
    """
    For the given node (and level for cluster mode), compute ΔE if we set
    that node/cluster to each bin. Returns dE per bin, current bin, best bin,
    true bin (perfect_bins-based), nodes (for cluster), and is_cluster flag.
    """
    be = backend
    if level is None:
        level_used = None
    elif be.weighted_clusters is not None and 0 <= int(level) < be.levels_available():
        level_used = int(level)
    else:
        level_used = None

    nodes, lvl, cid = be._nodes_for_action_target(node=node, level=level_used)
    if nodes.size == 0:
        return None

    current_bin = int(be._cluster_current_bin_weighted_mode(nodes))

    # True bin: weighted perfect_bins over nodes
    w = be.node_gal_counts[nodes].astype(np.float64)
    true_bins = perfect_bins[nodes].astype(np.int32)
    counts = np.bincount(true_bins, weights=w, minlength=be.K)
    true_bin = int(np.argmax(counts))

    dE_per_bin = np.full(be.K, np.nan, dtype=np.float64)
    for b in range(be.K):
        prop = be.make_proposal_set_bin(new_bin=b, node=node, level=level_used)
        if prop.get("type") == "noop":
            continue
        ev = be.evaluate_proposal_deltaE(prop)
        dE_per_bin[b] = float(ev["dE"])
        be.revert(ev["undo"])

    # Current bin: no move → dE = 0
    dE_per_bin[current_bin] = 0.0

    valid = np.isfinite(dE_per_bin)
    best_bin = int(np.argmax(dE_per_bin)) if np.any(valid) else current_bin

    return {
        "dE_per_bin": dE_per_bin,
        "current_bin": current_bin,
        "best_bin": best_bin,
        "true_bin": true_bin,
        "nodes": nodes,
        "is_cluster": lvl is not None,
    }

In [4]:
# Cache: level -> {cluster_id: stats}
# stats: dict(best_bin, best_dE, true_bin, nodes)
cluster_cache = {}


def compute_cluster_stats_for_level(level, backend, perfect_bins):
    level = int(level)
    if level in cluster_cache:
        return cluster_cache[level]

    if backend.weighted_clusters is None:
        raise RuntimeError("weighted_clusters not set on backend; cluster view requires hierarchy.")

    print(f"Computing cluster stats for level {level} (this may take a moment)...")

    cluster_grid = backend.cluster_grid(level)
    flat = cluster_grid.reshape(-1)
    max_cid = int(flat.max(initial=-1))
    stats = {}

    for cid in range(max_cid + 1):
        nodes = backend.cluster_nodes(level, cid)
        if nodes.size == 0:
            continue
        # representative node for this cluster (same pattern as env/backend)
        node0 = int(nodes[0])

        res = compute_deltaE_per_bin(backend, node0, level, perfect_bins)
        if res is None:
            continue

        dEs = res["dE_per_bin"]
        # best ΔE and corresponding bin (greedy cluster move)
        valid = np.isfinite(dEs)
        if not np.any(valid):
            best_dE = np.nan
            best_bin = res["current_bin"]
        else:
            best_idx = int(np.argmax(dEs))
            best_dE = float(dEs[best_idx])
            best_bin = best_idx

        stats[cid] = {
            "best_bin": best_bin,
            "best_dE": best_dE,
            "true_bin": int(res["true_bin"]),
            "nodes": res["nodes"].copy(),
        }

    cluster_cache[level] = stats
    print(f"Done level {level}: {len(stats)} clusters with galaxies.")
    return stats

In [19]:
from matplotlib.colors import ListedColormap


def run_floor_inspector(backend, perfect_bins):
    be = backend
    H, W, K = be.H, be.W, be.K

    # Widgets
    level_slider = widgets.IntSlider(
        value=max(LEVEL_MIN, be.levels_available() // 2),
        min=LEVEL_MIN,
        max=LEVEL_MAX,
        step=1,
        description="Level:",
        continuous_update=False,
        layout=widgets.Layout(width="400px"),
    )

    floor_slider = widgets.FloatLogSlider(
        value=1e-5,
        base=10,
        min=-9,   # 1e-7
        max=-3,   # 1e-2
        step=0.1,
        description="ΔE floor:",
        continuous_update=False,
        layout=widgets.Layout(width="400px"),
    )

    zero_floor_checkbox = widgets.Checkbox(
        value=False,
        description="Set floor = 0",
        indent=False,
        layout=widgets.Layout(width="150px"),
    )

    info_label = widgets.HTML(value="")

    # Figure: left = classification, right = reference bins (perfect SOM)
    plt.ioff()
    fig, (ax_class, ax_ref) = plt.subplots(
        1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [1, 1]}
    )
    # Equal space for both panels, leave right margin for colorbar so perfect SOM isn't cut off
    fig.subplots_adjust(left=0.02, right=0.88, wspace=0.02)

    # fixed reference: perfect bins as map
    perfect_grid = perfect_bins.reshape(H, W)
    im_ref = ax_ref.imshow(
        perfect_grid, vmin=0, vmax=K - 1, cmap="viridis", origin="upper", aspect="equal"
    )
    ax_ref.set_title("Perfect SOM bins (diagnostic)")
    ax_ref.set_xticks([])
    ax_ref.set_yticks([])
    cbar_ref = fig.colorbar(im_ref, ax=ax_ref, fraction=0.046, pad=0.04, shrink=0.8)
    cbar_ref.set_label("Bin")

    # classification cmap: 0=off/under-floor (grey), 1=bad (red), 2=good (green)
    class_cmap = ListedColormap(["#cccccc", "#e41a1c", "#4daf4a"])

    def update_plot(change=None):
        level = int(level_slider.value)
        # Either take slider value or force 0 depending on checkbox
        raw_floor = float(floor_slider.value)
        effective_floor = 0.0 if zero_floor_checkbox.value else raw_floor
        # keep UI consistent (disable slider when overridden)
        floor_slider.disabled = bool(zero_floor_checkbox.value)

        stats = compute_cluster_stats_for_level(level, be, perfect_bins)

        # 0=off (max dE < floor or NaN), 1=bad (best_bin != true), 2=good (best_bin == true)
        class_map = np.zeros((H, W), dtype=np.int8)

        # cluster counts
        n_good = 0
        n_bad = 0
        n_usable = 0
        # node-area counts (number of nodes in clusters)
        area_good = 0
        area_bad = 0
        area_usable = 0

        # cluster ids per node to paint whole grid
        cluster_grid = be.cluster_grid(level)  # (H,W) cluster id
        for cid, st in stats.items():
            best_dE = st["best_dE"]
            best_bin = st["best_bin"]
            true_bin = st["true_bin"]
            nodes = st["nodes"]
            n_nodes = int(nodes.size)

            if not np.isfinite(best_dE) or best_dE < effective_floor:
                cls = 0  # off / below floor
            else:
                n_usable += 1
                area_usable += n_nodes
                if best_bin == true_bin:
                    cls = 2
                    n_good += 1
                    area_good += n_nodes
                else:
                    cls = 1
                    n_bad += 1
                    area_bad += n_nodes

            # paint all nodes in this cluster
            class_map[cluster_grid == cid] = cls

        ax_class.clear()
        ax_class.imshow(
            class_map, vmin=0, vmax=2, cmap=class_cmap, origin="upper", aspect="equal"
        )
        ax_class.set_title(f"Level {level} – cluster quality vs ΔE floor")
        ax_class.set_xticks([])
        ax_class.set_yticks([])

        floor_for_label = effective_floor
        info_label.value = (
            f"<b>Level {level}</b>, floor = {floor_for_label:.1e} &nbsp; | "
            f"usable clusters: {n_usable} (good: {n_good}, bad: {n_bad})<br>"
            f"node area (good: {area_good}, bad: {area_bad}, usable: {area_usable})"
        )

        fig.canvas.draw_idle()

    level_slider.observe(update_plot, names="value")
    floor_slider.observe(update_plot, names="value")
    zero_floor_checkbox.observe(update_plot, names="value")

    # Initial draw
    update_plot()

    controls = widgets.VBox(
        [level_slider, widgets.HBox([floor_slider, zero_floor_checkbox]), info_label],
        layout=widgets.Layout(align_items="flex-start"),
    )
    display(widgets.VBox([controls, fig.canvas]))


run_floor_inspector(backend, perfect_bins)

Computing cluster stats for level 100 (this may take a moment)...
Done level 100: 9999 clusters with galaxies.


### Floor Tuner - Flortuna Dusseldorf

In [10]:
# --- Floor tuning algorithm ---
# Maximise good clusters, minimise bad, prefer lower floor; constraint: floor(level) < floor(level-1)

def count_good_bad_usable(backend, perfect_bins, level, floor):
    """For a given (level, floor), return cluster and node-area stats.

    Returns:
      (n_good, n_bad, n_usable, area_good, area_bad, area_usable)
    where counts are in clusters and areas are in nodes.
    """
    stats = compute_cluster_stats_for_level(level, backend, perfect_bins)
    cluster_grid = backend.cluster_grid(level)
    n_good = n_bad = n_usable = 0
    area_good = area_bad = area_usable = 0
    for cid, st in stats.items():
        best_dE = st["best_dE"]
        best_bin = st["best_bin"]
        true_bin = st["true_bin"]
        nodes = st["nodes"]
        n_nodes = int(nodes.size)
        if not np.isfinite(best_dE) or best_dE < floor:
            continue
        n_usable += 1
        area_usable += n_nodes
        if best_bin == true_bin:
            n_good += 1
            area_good += n_nodes
        else:
            n_bad += 1
            area_bad += n_nodes
    return n_good, n_bad, n_usable, area_good, area_bad, area_usable


def tune_floor_profile(
    backend,
    perfect_bins,
    level_min,
    level_max,
    *,
    alpha=1.0,
    margin_ratio=0.2,
    floor_lo=1e-7,
    floor_hi=1e-2,
    n_floor_steps=40,
):
    """
    Tune floor per level using node-area stats:

      - Primary: favour good node area over bad node area via
            primary = area_good - alpha * area_bad
      - Margin constraint (preferred when possible):
            area_good >= (1 + margin_ratio) * area_bad
        so good area clearly dominates bad.
      - Among floors at a level, scan from high -> low and pick the
        highest floor that satisfies the margin. If none do, fall back
        to the floor with best `primary`.
      - Maintain monotone constraint: floor(level) < floor(level-1).

    Returns:
      profile: dict[int, float] level -> chosen floor
      details: dict[level] -> stats dict with counts/areas and constraint flag
    """
    profile = {}
    details = {}

    # log-spaced candidate floors, same for every level; we subset per level
    base_grid = np.logspace(np.log10(floor_lo), np.log10(floor_hi), n_floor_steps)

    for level in range(level_min, level_max + 1):
        # enforce monotone: candidates must be < previous floor
        if level > level_min:
            prev_floor = profile[level - 1]
            candidates = base_grid[base_grid < prev_floor]
            if candidates.size == 0:
                candidates = np.array([prev_floor * 0.5])
        else:
            prev_floor = None
            candidates = base_grid

        # scan from high floor -> low floor to avoid collapsing unnecessarily
        candidates_sorted = np.sort(candidates)[::-1]

        feasible_choice = None   # (floor, counts_tuple, primary)
        best_relaxed = None      # (floor, counts_tuple, primary)

        for floor in candidates_sorted:
            n_good, n_bad, n_usable, area_good, area_bad, area_usable = count_good_bad_usable(
                backend, perfect_bins, level, float(floor)
            )
            primary = float(area_good - alpha * area_bad)

            # record best relaxed (no constraint)
            if (best_relaxed is None) or (primary > best_relaxed[2]):
                best_relaxed = (
                    float(floor),
                    (n_good, n_bad, n_usable, area_good, area_bad, area_usable),
                    primary,
                )

            # margin constraint: good area must exceed bad area by factor (1+margin_ratio)
            if area_bad > 0 and area_good < (1.0 + margin_ratio) * area_bad:
                continue

            # first floor in high->low order that passes margin is our feasible choice
            feasible_choice = (
                float(floor),
                (n_good, n_bad, n_usable, area_good, area_bad, area_usable),
                primary,
            )
            break

        if feasible_choice is not None:
            chosen_floor, counts, primary = feasible_choice
            constraint_flag = "feasible"
        else:
            chosen_floor, counts, primary = best_relaxed
            constraint_flag = "relaxed"

        profile[level] = chosen_floor
        n_good, n_bad, n_usable, area_good, area_bad, area_usable = counts

        details[level] = {
            "level": level,
            "floor": chosen_floor,
            "n_good": n_good,
            "n_bad": n_bad,
            "n_usable": n_usable,
            "area_good": area_good,
            "area_bad": area_bad,
            "area_usable": area_usable,
            "primary": primary,
            "constraint": constraint_flag,
            "prev_floor": prev_floor,
        }

        print(
            f"Level {level} [{constraint_flag}]: floor={chosen_floor:.2e}  "
            f"good={n_good} bad={n_bad} usable={n_usable} | "
            f"area_good={area_good} area_bad={area_bad} area_usable={area_usable}"
        )

    return profile, details

In [16]:
# Run the tuner and save the profile (takes a few minutes: one level sweep per level)

profile, details = tune_floor_profile(
    backend,
    perfect_bins,
    LEVEL_MIN,
    LEVEL_MAX,
    alpha=1.0,      # weight for penalising bad clusters
    margin_ratio=0.5,
    floor_lo=1e-7,
    floor_hi=1e-4,
    n_floor_steps=40,
)

# Save to data/ so it can be loaded by the env
from som_rl.env.floor_profile import save_floor_profile, load_floor_profile, floor_schedule_from_profile

profile_path = f"{DATA}/floor_profile.json"
save_floor_profile(profile, profile_path, level_min=LEVEL_MIN, level_max=LEVEL_MAX, format="json")
print(f"Saved profile to {profile_path}")

# Loader usage in training / env config:
#   from som_rl.env.floor_profile import floor_schedule_from_profile
#   energy_floor = floor_schedule_from_profile("data/floor_profile.json")
#   cfg = SOMEnergyEnvConfig(energy_floor=energy_floor, ...)

Level 30 [feasible]: floor=1.00e-04  good=36 bad=13 usable=49 | area_good=1820 area_bad=706 area_usable=2526
Level 31 [feasible]: floor=8.38e-05  good=40 bad=14 usable=54 | area_good=1872 area_bad=705 area_usable=2577
Level 32 [feasible]: floor=7.02e-05  good=46 bad=16 usable=62 | area_good=2034 area_bad=801 area_usable=2835
Level 33 [feasible]: floor=5.88e-05  good=49 bad=18 usable=67 | area_good=2041 area_bad=850 area_usable=2891
Level 34 [feasible]: floor=4.92e-05  good=51 bad=19 usable=70 | area_good=2033 area_bad=856 area_usable=2889
Level 35 [feasible]: floor=4.12e-05  good=58 bad=21 usable=79 | area_good=2149 area_bad=959 area_usable=3108
Level 36 [feasible]: floor=3.46e-05  good=61 bad=22 usable=83 | area_good=2106 area_bad=962 area_usable=3068
Level 37 [feasible]: floor=2.89e-05  good=63 bad=25 usable=88 | area_good=2121 area_bad=1024 area_usable=3145
Level 38 [feasible]: floor=2.42e-05  good=65 bad=31 usable=96 | area_good=1956 area_bad=1194 area_usable=3150
Level 39 [feasibl